# 实验 5：Agentic RAG 与外部记忆

## 准备工作<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"><p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 和 <code>helper.py</code> 文件：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Open"</em>。<p> ⬇ &nbsp; <b>下载笔记本：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p><p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"附录 – 提示、帮助和下载"</em> 课程。</p></div>

## 第 0 部分：设置 Letta 客户端

In [1]:
from letta_client import Letta

client = Letta(base_url="http://localhost:8283")

In [2]:
def print_message(message):  
    if message.message_type == "reasoning_message": 
        print("🧠 Reasoning: " + message.reasoning) 
    elif message.message_type == "assistant_message": 
        print("🤖 Agent: " + message.content) 
    elif message.message_type == "tool_call_message": 
        print("🔧 Tool Call: " + message.tool_call.name + "\n" + message.tool_call.arguments)
    elif message.message_type == "tool_return_message": 
        print("🔧 Tool Return: " + message.tool_return)
    elif message.message_type == "user_message": 
        print("👤 User Message: " + message.content)

## 第 1 部分：数据源

### 创建数据源

In [3]:
source = client.sources.create(
    name="employee_handbook",
    embedding="openai/text-embedding-3-small"
)
source

Source(id='source-a97dff3f-1133-4470-a0c2-3418a4de30d5', name='employee_handbook', description=None, embedding_config=EmbeddingConfig(embedding_endpoint_type='openai', embedding_endpoint='http://jupyter-api-proxy.internal.dlai/rev-proxy/letta', embedding_model='text-embedding-3-small', embedding_dim=2000, embedding_chunk_size=300, handle='openai/text-embedding-3-small', azure_endpoint=None, azure_version=None, azure_deployment=None), metadata=None, created_by_id='user-00000000-0000-4000-8000-000000000000', last_updated_by_id='user-00000000-0000-4000-8000-000000000000', created_at=datetime.datetime(2026, 4, 29, 7, 35, 53), updated_at=datetime.datetime(2026, 4, 29, 7, 35, 53), organization_id='org-00000000-0000-4000-8000-000000000000')

### 上传数据源

In [4]:
job = client.sources.files.upload(
    source_id=source.id,
    file=open("handbook.pdf", "rb")
)

In [5]:
job.status

'created'

### 查看任务状态随时间的变化

In [6]:
import time
from letta_client import JobStatus

while job.status != 'completed':
    job = client.jobs.retrieve(job.id)
    print(job.status)
    time.sleep(1)

running
running
running
running
completed


### 查看任务元数据

In [7]:
job.metadata

{'type': 'embedding',
 'filename': 'handbook.pdf',
 'source_id': 'source-a97dff3f-1133-4470-a0c2-3418a4de30d5',
 'num_passages': 11,
 'num_documents': 1}

In [8]:
passages = client.sources.passages.list(
    source_id=source.id,
)
len(passages)

11

### 创建一个代理并附加数据源

In [9]:
agent_state = client.agents.create(
    memory_blocks=[
        {
          "label": "human",
          "value": "My name is Sarah"
        },
        {
          "label": "persona",
          "value": "You are a helpful assistant"
        }
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-3-small"
)

In [10]:
agent_state = client.agents.sources.attach(
    agent_id=agent_state.id, 
    source_id=source.id
)

### 查看代理已附加的数据源

In [11]:
client.agents.sources.list(agent_id=agent_state.id)

[Source(id='source-a97dff3f-1133-4470-a0c2-3418a4de30d5', name='employee_handbook', description=None, embedding_config=EmbeddingConfig(embedding_endpoint_type='openai', embedding_endpoint='http://jupyter-api-proxy.internal.dlai/rev-proxy/letta', embedding_model='text-embedding-3-small', embedding_dim=2000, embedding_chunk_size=300, handle='openai/text-embedding-3-small', azure_endpoint=None, azure_version=None, azure_deployment=None), metadata=None, created_by_id='user-00000000-0000-4000-8000-000000000000', last_updated_by_id='user-00000000-0000-4000-8000-000000000000', created_at=datetime.datetime(2026, 4, 29, 7, 35, 53), updated_at=datetime.datetime(2026, 4, 29, 7, 35, 53), organization_id='org-00000000-0000-4000-8000-000000000000')]

In [12]:
passages = client.agents.passages.list(agent_id=agent_state.id)
len(passages)

11

### 与代理交互并引用已附加的数据源

In [13]:
response = client.agents.messages.create(
    agent_id=agent_state.id,
    messages=[
        {
            "role": "user",
            "content": "Search archival for our company's vacation policies"
        }
    ]
)
for message in response.messages:
    print_message(message)

🧠 Reasoning: User is requesting information on vacation policies. I need to search the archival memory for relevant details.
🔧 Tool Call: archival_memory_search
{
  "query": "vacation policy",
  "page": 0,
  "start": 0,
  "request_heartbeat": true
}
🔧 Tool Return: ([{'timestamp': '2026-04-29 07:35:57.957742', 'content': "vacationsarepermittedonlyunderthefollowingcondition: youmust provideanAI agent that matchesorsurpassesyourowncompetenciestofullyperformyourdutiesduringyourabsence.\nTheAI replacement must beequivalentlycompetent inall aspectsof yourrole, ensuringseamlesscontinuityof operations. Youarerequiredtosubmit theAI agent toyourimmediatesupervisorat least fourweekspriortoyourintendedleavedate. Thistimeframeallowsforrigoroustestingandevaluationof theAI'scapabilitiesandreliability.\nTheAI will undergocomprehensiveassessmentstoverifyitsproficiencyandeffectivenessinhandlingyourresponsibilities. Approval of theAI agent isat thesolediscretionof uppermanagement, andsubmissiondoesnot gu

## 第 2 部分：使用自定义工具连接数据

### 创建自定义工具

In [14]:
def query_birthday_db(name: str):
    """
    This tool queries an external database to
    lookup the birthday of someone given their name.

    Args:
        name (str): The name to look up

    Returns:
        birthday (str): The birthday in mm-dd-yyyy format

    """
    my_fake_data = {
        "bob": "03-06-1997",
        "sarah": "07-06-1993"
    }
    name = name.lower()
    if name not in my_fake_data:
        return None
    else:
        return my_fake_data[name]

In [15]:
birthday_tool = client.tools.upsert_from_function(func=query_birthday_db)

### 创建一个可访问工具的代理

In [16]:
agent_state = client.agents.create(
    memory_blocks=[
        {
          "label": "human",
          "value": "My name is Sarah"
        },
        {
          "label": "persona",
          "value": "You are a agent with access to a birthday_db " \
            + "that you use to lookup information about users' birthdays."
        }
    ],
    model="openai/gpt-4o-mini-2024-07-18",
    embedding="openai/text-embedding-3-small",
    tool_ids=[birthday_tool.id],
    #tool_exec_environment_variables={"DB_KEY": "my_key"}
)

In [17]:
# 向代理发送一条消息response = client.agents.messages.create_stream(    agent_id=agent_state.id,    messages=[        {            "role": "user",            "content": "whens my bday????"        }    ])for message in response:    print_message(message)

🧠 Reasoning: User is asking about their birthday. Looking up in the birthday database.
🔧 Tool Call: query_birthday_db
{
  "name": "Sarah",
  "request_heartbeat": true
}
🔧 Tool Return: 07-06-1993
🧠 Reasoning: Birthday found. Preparing to share the date with Sarah.
🤖 Agent: Your birthday is on July 6, 1993! 🎉
